In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

# ── Load core data sources ─────────────────────────────────────────────
DATA_DIR = '/home/h604827/ControlActions/DATA'
TARGET_PV_COL = '03LIC_1071.PV'

events_df = pd.read_csv(f'{DATA_DIR}/trip_filtered_events.csv', low_memory=False)
events_df['VT_Start'] = pd.to_datetime(events_df['VT_Start'])
events_df = events_df.sort_values('VT_Start').reset_index(drop=True)

ts_df = pd.read_parquet(f'{DATA_DIR}/03LIC_1071_JAN_2026.parquet')
ts_df['TimeStamp'] = pd.to_datetime(ts_df['TimeStamp'])
ts_df = ts_df.set_index('TimeStamp').sort_index()

op_limits = pd.read_csv(f'{DATA_DIR}/operating_limits.csv')
target_limit_row = op_limits.loc[op_limits['TAG_NAME'] == TARGET_PV_COL].squeeze()
TARGET_LOWER_LIMIT = float(target_limit_row['LOWER_LIMIT'])
TARGET_UPPER_LIMIT = float(target_limit_row['UPPER_LIMIT'])

print(f"Events:      {len(events_df):,} rows, {events_df['VT_Start'].min().date()} to {events_df['VT_Start'].max().date()}")
print(f"Time series: {len(ts_df):,} rows, {ts_df.index.min().date()} to {ts_df.index.max().date()}")
print(f"Target operating limits for {TARGET_PV_COL}: {TARGET_LOWER_LIMIT:.2f} to {TARGET_UPPER_LIMIT:.2f}")

Events:      1,600,520 rows, 2021-09-06 to 2025-06-27
Time series: 1,737,586 rows, 2022-01-03 to 2025-06-23
Target operating limits for 03LIC_1071.PV: 35.25 to 42.41


In [2]:
# ── Extract OP/SP CHANGE events for tags with OP in time series ────────
# These are operator actions that change the controller output or setpoint

# Tags that have OP columns in the time series
op_cols = [c for c in ts_df.columns if c.endswith('.OP')]
op_tag_names = [c.replace('.OP', '') for c in op_cols]

# Filter CHANGE events: OP or SP changes with valid PrevValue
change_events = events_df[
    (events_df['ConditionName'] == 'CHANGE') &
    (events_df['Description'].isin(['OP', 'SP'])) &
    (events_df['Source'].isin(op_tag_names))
].copy()

# Remove duplicates (same timestamp, keep the one with PrevValue)
dup_mask = change_events.duplicated(subset=['VT_Start', 'Source', 'Description'], keep=False)
change_events = change_events[~(dup_mask & change_events['PrevValue'].isna())]

# Convert to numeric
change_events['PrevValue'] = pd.to_numeric(change_events['PrevValue'], errors='coerce')
change_events['Value'] = pd.to_numeric(change_events['Value'], errors='coerce')
change_events = change_events.dropna(subset=['PrevValue', 'Value'])

# Compute change magnitude
change_events['delta'] = change_events['Value'] - change_events['PrevValue']
change_events['abs_delta'] = change_events['delta'].abs()

# Drop trivial changes (delta == 0)
change_events = change_events[change_events['abs_delta'] > 0].copy()

print(f"Total OP/SP change events (non-zero delta): {len(change_events):,}")
print(f"\nPer-tag breakdown:")
print(change_events.groupby('Source').size().sort_values(ascending=False).to_string())

Total OP/SP change events (non-zero delta): 23,750

Per-tag breakdown:
Source
03PIC_1013    7390
03FIC_3415    5575
03LIC_1085    2959
03PIC_3131    1834
03LIC_1071    1401
03LIC_1097     905
03LIC_1094     889
03FIC_1085     837
03LIC_1016     756
03LIC_3178     684
03PIC_1068     481
03PIC_1104      34
03TIC_1092       5


# Response Time And Settling Analysis: OP Actions To 03LIC_1071.PV

## Scope
This notebook now focuses only on the SME questions around operator actions:

1. How long after an OP or SP change does `03LIC_1071.PV` start responding?
2. How long does it take to settle?
3. Was the control action an increase or a decrease, and how large was it?
4. What was `03LIC_1071.PV` when the action was taken?
5. What value did `03LIC_1071.PV` settle around, and was that inside or outside its operating limits?

## Method
- Use isolated OP or SP change events so each PV response can be attributed to one action.
- Measure baseline variability from the 10 minutes before the action.
- Measure response time from the first full minute after the action.
- Define settling time as the first sustained period where rolling PV standard deviation falls below 50% of the baseline standard deviation.
- Record the settled PV as the mean of the first sustained settled segment and compare it with the operating limits of `03LIC_1071.PV`.

In [3]:
# ── Identify isolated OP changes ──────────────────────────────────────
# An "isolated" change means no other OP/SP change on ANY monitored tag
# occurs within ±ISOLATION_WINDOW minutes. This lets us attribute PV
# response to a single operator action.

ISOLATION_WINDOW_MINS = 15  # minutes before & after to require isolation
TARGET_PV_COL = '03LIC_1071.PV'

# Sort all change events by time
change_events_sorted = change_events.sort_values('VT_Start').reset_index(drop=True)

# For each event, find the time gap to the previous and next event (across ALL tags)
change_events_sorted['time_to_prev'] = change_events_sorted['VT_Start'].diff().dt.total_seconds() / 60
change_events_sorted['time_to_next'] = change_events_sorted['VT_Start'].diff(-1).dt.total_seconds().abs() / 60

# Mark as isolated if both gaps > ISOLATION_WINDOW_MINS
isolated_mask = (
    (change_events_sorted['time_to_prev'] > ISOLATION_WINDOW_MINS) |
    (change_events_sorted['time_to_prev'].isna())  # first event
) & (
    (change_events_sorted['time_to_next'] > ISOLATION_WINDOW_MINS) |
    (change_events_sorted['time_to_next'].isna())  # last event
)

isolated_events = change_events_sorted[isolated_mask].copy()
print(f"Isolated OP/SP changes (±{ISOLATION_WINDOW_MINS}min window): {len(isolated_events):,} / {len(change_events_sorted):,}")
print(f"\nPer-tag breakdown of isolated events:")
print(isolated_events.groupby('Source').size().sort_values(ascending=False).to_string())

Isolated OP/SP changes (±15min window): 1,227 / 23,750

Per-tag breakdown of isolated events:
Source
03LIC_1085    410
03PIC_1013    316
03FIC_3415    254
03LIC_1097     56
03LIC_1071     51
03LIC_1016     34
03LIC_3178     29
03PIC_1068     27
03PIC_3131     22
03LIC_1094     14
03FIC_1085      6
03PIC_1104      5
03TIC_1092      3


In [4]:
# ── Measure response time and settling outcome for each isolated OP change ──
PRE_WINDOW = 10
POST_WINDOW = 60
RESPONSE_THRESHOLD_FACTOR = 0.5
SETTLING_STD_WINDOW = 5
SETTLING_STD_FACTOR = 0.5
SETTLED_VALUE_WINDOW = 5

def classify_limit_zone(value, lower_limit, upper_limit):
    if pd.isna(value):
        return 'unknown'
    if value < lower_limit:
        return 'below_lower_limit'
    if value > upper_limit:
        return 'above_upper_limit'
    return 'within_limits'

def distance_outside_limits(value, lower_limit, upper_limit):
    if pd.isna(value):
        return np.nan
    if value < lower_limit:
        return lower_limit - value
    if value > upper_limit:
        return value - upper_limit
    return 0.0

results = []

for _, row in isolated_events.iterrows():
    t0 = row['VT_Start']
    tag = row['Source']
    desc = row['Description']
    op_delta = row['delta']
    t0_rounded = t0.floor('min')

    pre_start = t0_rounded - timedelta(minutes=PRE_WINDOW)
    pre_end = t0_rounded
    post_end = t0_rounded + timedelta(minutes=POST_WINDOW)

    baseline = ts_df.loc[pre_start:pre_end, TARGET_PV_COL].dropna()
    post_pv = ts_df.loc[t0_rounded:post_end, TARGET_PV_COL].dropna()

    if len(baseline) < 3 or len(post_pv) < SETTLING_STD_WINDOW:
        continue

    baseline_mean = baseline.mean()
    baseline_std = baseline.std()
    if pd.isna(baseline_std) or baseline_std < 0.1:
        baseline_std = 0.1

    pv_at_action = post_pv.iloc[0]
    post_action_pv = post_pv.iloc[1:].copy()
    if len(post_action_pv) < SETTLING_STD_WINDOW:
        continue

    response_threshold = RESPONSE_THRESHOLD_FACTOR * baseline_std
    pv_deviation = (post_action_pv - baseline_mean).abs()
    response_candidates = pv_deviation[pv_deviation > response_threshold].index
    response_time_mins = (response_candidates[0] - t0_rounded).total_seconds() / 60 if len(response_candidates) > 0 else np.nan

    rolling_std = post_action_pv.rolling(window=SETTLING_STD_WINDOW, min_periods=3).std()
    settle_threshold = SETTLING_STD_FACTOR * baseline_std
    settled_mask = (rolling_std < settle_threshold).fillna(False)

    settling_time_mins = np.nan
    settled_pv = np.nan
    settled_pv_std = np.nan
    settled_start_time = pd.NaT
    settled_end_time = pd.NaT

    settled_groups = settled_mask.astype(int).groupby((settled_mask.astype(int) != settled_mask.astype(int).shift()).cumsum())
    for _, grp in settled_groups:
        if grp.iloc[0] != 1 or len(grp) < SETTLED_VALUE_WINDOW:
            continue
        settled_start_time = grp.index[0]
        settled_end_time = grp.index[SETTLED_VALUE_WINDOW - 1]
        settled_slice = post_action_pv.loc[settled_start_time:settled_end_time].dropna()
        if len(settled_slice) < 3:
            continue
        settling_time_mins = (settled_start_time - t0_rounded).total_seconds() / 60
        settled_pv = settled_slice.mean()
        settled_pv_std = settled_slice.std()
        break

    action_zone = classify_limit_zone(pv_at_action, TARGET_LOWER_LIMIT, TARGET_UPPER_LIMIT)
    settled_zone = classify_limit_zone(settled_pv, TARGET_LOWER_LIMIT, TARGET_UPPER_LIMIT)
    action_outside_distance = distance_outside_limits(pv_at_action, TARGET_LOWER_LIMIT, TARGET_UPPER_LIMIT)
    settled_outside_distance = distance_outside_limits(settled_pv, TARGET_LOWER_LIMIT, TARGET_UPPER_LIMIT)
    movement_towards_limits = action_outside_distance - settled_outside_distance if pd.notna(settled_outside_distance) else np.nan

    op_col = f'{tag}.OP'
    op_value_at_change = np.nan
    if op_col in ts_df.columns and t0_rounded in ts_df.index:
        op_value_at_change = ts_df.at[t0_rounded, op_col]

    results.append({
        'event_time': t0,
        'event_time_rounded': t0_rounded,
        'source_tag': tag,
        'change_type': desc,
        'op_delta': op_delta,
        'abs_op_delta': abs(op_delta),
        'action_direction': 'increase' if op_delta > 0 else 'decrease',
        'op_value_at_change': op_value_at_change,
        'pv_baseline_mean': baseline_mean,
        'pv_baseline_std': baseline_std,
        'pv_at_action': pv_at_action,
        'action_zone': action_zone,
        'action_outside_distance': action_outside_distance,
        'response_time_mins': response_time_mins,
        'settling_time_mins': settling_time_mins,
        'settled_pv': settled_pv,
        'settled_pv_std': settled_pv_std,
        'settled_start_time': settled_start_time,
        'settled_end_time': settled_end_time,
        'settled_zone': settled_zone,
        'settled_outside_distance': settled_outside_distance,
        'movement_towards_limits': movement_towards_limits,
        'moved_closer_to_limits': movement_towards_limits > 0 if pd.notna(movement_towards_limits) else False,
        'pv_change_to_settled': settled_pv - pv_at_action if pd.notna(settled_pv) else np.nan,
        'year': t0.year,
    })

response_df = pd.DataFrame(results)
response_df['settled_within_limits'] = response_df['settled_zone'] == 'within_limits'

print(f"Computed metrics for {len(response_df):,} isolated events")
print(f"Events with measurable response: {response_df['response_time_mins'].notna().sum():,}")
print(f"Events that settled within {POST_WINDOW} minutes: {response_df['settling_time_mins'].notna().sum():,}")
print()
print('Action direction split:')
print(response_df['action_direction'].value_counts().to_string())
print()
print('Settled zone split:')
print(response_df['settled_zone'].value_counts(dropna=False).to_string())

Computed metrics for 1,096 isolated events
Events with measurable response: 1,093
Events that settled within 60 minutes: 758

Action direction split:
action_direction
decrease    575
increase    521

Settled zone split:
settled_zone
within_limits        398
unknown              338
below_lower_limit    212
above_upper_limit    148


## SME-Focused Outputs

The outputs below are intentionally limited to the operational questions:

- Which tag was moved, in which direction, and by how much?
- What was `03LIC_1071.PV` at the time of the action?
- How long did it take to respond and settle?
- What value did it settle around?
- Did it settle back inside the operating limits?

In [5]:
# ── Summary table for SME review ───────────────────────────────────────
summary_df = response_df.groupby(['source_tag', 'action_direction']).agg(
    n_events=('event_time', 'count'),
    min_op_change=('abs_op_delta', 'min'),
    median_op_change=('abs_op_delta', 'median'),
    mean_op_change=('abs_op_delta', 'mean'),
    max_op_change=('abs_op_delta', 'max'),
    median_response_time_min=('response_time_mins', 'median'),
    median_settling_time_min=('settling_time_mins', 'median'),
    median_pv_at_action=('pv_at_action', 'median'),
    median_settled_pv=('settled_pv', 'median'),
    pct_settled_within_limits=('settled_within_limits', lambda s: 100 * s.mean()),
    median_limit_improvement=('movement_towards_limits', 'median'),
).reset_index()

summary_df = summary_df.sort_values(['n_events', 'source_tag', 'action_direction'], ascending=[False, True, True])
display(summary_df.round(2))

event_review_cols = [
    'event_time', 'source_tag', 'change_type', 'action_direction', 'op_delta',
    'pv_at_action', 'action_zone', 'response_time_mins', 'settling_time_mins',
    'settled_pv', 'settled_zone', 'movement_towards_limits'
]
event_review_df = response_df[event_review_cols].sort_values('event_time', ascending=False)
display(event_review_df.head(20).round(2))

,source_tag,action_direction,n_events,min_op_change,median_op_change,mean_op_change,max_op_change,median_response_time_min,median_settling_time_min,median_pv_at_action,median_settled_pv,pct_settled_within_limits,median_limit_improvement
9,03LIC_1085,increase,181,0.05,1.00,0.86,2.00,1.0,11.0,38.44,37.27,36.46,0.00
16,03PIC_1013,decrease,166,0.47,2.00,1.61,4.99,1.0,12.0,38.96,39.37,39.16,0.00
8,03LIC_1085,decrease,165,0.10,1.00,0.96,3.00,1.0,10.0,36.82,36.88,35.76,0.00
2,03FIC_3415,decrease,150,1.00,3.00,4.83,75.00,1.0,9.0,39.19,39.45,36.00,0.00
17,03PIC_1013,increase,132,0.39,1.50,1.48,2.02,1.0,17.0,38.60,38.14,37.12,0.00
3,03FIC_3415,increase,103,1.00,3.00,4.60,27.00,1.0,11.0,40.22,40.24,35.92,0.00
7,03LIC_1071,increase,34,0.50,1.00,1.52,4.00,1.0,18.0,29.57,35.72,17.65,1.40
12,03LIC_1097,decrease,31,0.20,1.00,3.12,35.00,1.0,13.0,38.00,36.02,54.84,0.00
5,03LIC_1016,increase,20,0.50,1.00,1.30,2.00,1.0,12.5,35.18,35.69,40.00,0.02
13,03LIC_1097,increase,18,0.06,1.00,2.53,23.00,1.0,13.5,36.19,35.72,22.22,-0.79


,event_time,source_tag,change_type,action_direction,op_delta,pv_at_action,action_zone,response_time_mins,settling_time_mins,settled_pv,settled_zone,movement_towards_limits
1095,2025-06-23 13:34:23.641500,03PIC_1013,OP,decrease,-2.00,43.22,above_upper_limit,1.0,NaN,NaN,unknown,NaN
1094,2025-06-22 17:03:51.553700,03PIC_1013,OP,decrease,-2.00,45.96,above_upper_limit,1.0,NaN,NaN,unknown,NaN
1093,2025-06-22 16:04:26.492700,03LIC_1071,SP,increase,2.00,32.39,below_lower_limit,1.0,NaN,NaN,unknown,NaN
1092,2025-06-22 14:56:29.897100,03LIC_1071,SP,increase,2.00,36.56,within_limits,1.0,NaN,NaN,unknown,NaN
1091,2025-06-21 22:15:54.527500,03LIC_1071,SP,increase,1.00,29.34,below_lower_limit,1.0,3.0,34.80,below_lower_limit,5.46
1090,2025-06-21 12:26:47.033900,03LIC_1085,SP,increase,1.00,39.41,within_limits,1.0,NaN,NaN,unknown,NaN
1089,2025-06-18 16:45:12.690700,03PIC_1013,OP,increase,2.00,31.44,below_lower_limit,2.0,40.0,43.05,above_upper_limit,3.18
1088,2025-06-18 16:13:41.486400,03PIC_1013,OP,increase,2.00,38.59,within_limits,1.0,NaN,NaN,unknown,NaN
1087,2025-06-18 08:48:13.775300,03LIC_1097,SP,decrease,-1.00,38.76,within_limits,1.0,NaN,NaN,unknown,NaN
1086,2025-06-17 14:25:51.200800,03PIC_1013,OP,decrease,-1.02,45.84,above_upper_limit,2.0,NaN,NaN,unknown,NaN


In [9]:
# ── Visual 3: Settling-time histogram with OP-magnitude stats ──────────
hist_df = response_df.dropna(subset=['settling_time_mins', 'abs_op_delta']).copy()
tag_counts = hist_df['source_tag'].value_counts()
hist_tags = tag_counts[tag_counts >= 10].index.tolist()
hist_df = hist_df[hist_df['source_tag'].isin(hist_tags)].copy()

BIN_WIDTH = 5
max_settling = hist_df['settling_time_mins'].max()
bin_edges = np.arange(0, np.ceil(max_settling / BIN_WIDTH) * BIN_WIDTH + BIN_WIDTH, BIN_WIDTH)
hist_df['settling_bin'] = pd.cut(
    hist_df['settling_time_mins'],
    bins=bin_edges,
    right=False,
    include_lowest=True,
 )
hist_df = hist_df.dropna(subset=['settling_bin']).copy()
hist_df['bin_start'] = hist_df['settling_bin'].map(lambda interval: float(interval.left)).astype(float)
hist_df['bin_end'] = hist_df['bin_start'] + BIN_WIDTH
hist_df['bin_label'] = hist_df.apply(
    lambda row: f"{int(row['bin_start'])}-{int(row['bin_end'])} min", axis=1
 )

hist_summary = hist_df.groupby(['source_tag', 'action_direction', 'bin_start', 'bin_end', 'bin_label']).agg(
    n_events=('event_time', 'count'),
    median_settling_time=('settling_time_mins', 'median'),
    min_op_change=('abs_op_delta', 'min'),
    median_op_change=('abs_op_delta', 'median'),
    mean_op_change=('abs_op_delta', 'mean'),
    max_op_change=('abs_op_delta', 'max'),
).reset_index()

hist_summary['panel'] = hist_summary['source_tag'] + ' | ' + hist_summary['action_direction'].str.title()
panel_order = []
for tag in hist_tags:
    for direction in ['increase', 'decrease']:
        panel_name = f'{tag} | {direction.title()}'
        if panel_name in hist_summary['panel'].values:
            panel_order.append(panel_name)

fig = px.bar(
    hist_summary,
    x='bin_start',
    y='n_events',
    facet_col='panel',
    facet_col_wrap=4,
    category_orders={'panel': panel_order},
    custom_data=[
        'source_tag', 'action_direction', 'bin_label', 'n_events',
        'min_op_change', 'median_op_change', 'mean_op_change', 'max_op_change',
        'median_settling_time'
    ],
    title='Settling-Time Histogram By Tag And Action Direction',
    labels={
        'bin_start': 'Settling-time bin start (minutes)',
        'n_events': 'Number of actions',
        'panel': 'Tag and direction'
    }
)

fig.update_traces(
    hovertemplate=(
        'Tag: %{customdata[0]}<br>'
        'Direction: %{customdata[1]}<br>'
        'Settling-time bin: %{customdata[2]}<br>'
        'Actions in bin: %{customdata[3]}<br>'
        'Median settling time: %{customdata[8]:.1f} min<br>'
        'Min |OP change|: %{customdata[4]:.2f}<br>'
        'Median |OP change|: %{customdata[5]:.2f}<br>'
        'Mean |OP change|: %{customdata[6]:.2f}<br>'
        'Max |OP change|: %{customdata[7]:.2f}<extra></extra>'
    )
)

fig.update_xaxes(
    tickmode='linear',
    dtick=BIN_WIDTH,
    title_text='Settling time (minutes)'
)
fig.update_yaxes(title_text='Number of actions')
fig.update_layout(height=max(600, 260 * int(np.ceil(max(len(panel_order), 1) / 4))))
fig.show()

In [10]:
response_df

,event_time,event_time_rounded,source_tag,change_type,op_delta,abs_op_delta,action_direction,op_value_at_change,pv_baseline_mean,pv_baseline_std,...,settled_pv_std,settled_start_time,settled_end_time,settled_zone,settled_outside_distance,movement_towards_limits,moved_closer_to_limits,pv_change_to_settled,year,settled_within_limits
0,2022-01-05 10:20:47.603000,2022-01-05 10:20:00,03LIC_1085,SP,-1.0000,1.0000,decrease,47.703510,37.215811,0.818510,...,NaN,NaT,NaT,unknown,NaN,NaN,False,NaN,2022,False
1,2022-01-05 12:58:15.353200,2022-01-05 12:58:00,03LIC_1016,SP,0.9900,0.9900,increase,22.338234,38.450933,6.326280,...,1.077897,2022-01-05 13:01:00,2022-01-05 13:05:00,below_lower_limit,2.538113,2.824917,True,2.824917,2022,False
2,2022-01-07 14:55:10.169400,2022-01-07 14:55:00,03LIC_1016,SP,1.0000,1.0000,increase,33.035595,35.240398,4.419172,...,0.406305,2022-01-07 15:13:00,2022-01-07 15:17:00,below_lower_limit,0.864486,6.664970,True,6.664970,2022,False
3,2022-01-08 18:15:58.763300,2022-01-08 18:15:00,03LIC_1085,SP,-1.4912,1.4912,decrease,48.987880,37.894391,1.959319,...,0.224941,2022-01-08 18:18:00,2022-01-08 18:22:00,within_limits,0.000000,0.000000,False,1.385776,2022,True
4,2022-01-08 18:36:07.756200,2022-01-08 18:36:00,03LIC_1085,SP,-1.0000,1.0000,decrease,54.695786,35.868113,0.659372,...,NaN,NaT,NaT,unknown,NaN,NaN,False,NaN,2022,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1091,2025-06-21 22:15:54.527500,2025-06-21 22:15:00,03LIC_1071,SP,1.0000,1.0000,increase,64.929130,35.339375,3.281619,...,0.957743,2025-06-21 22:18:00,2025-06-21 22:22:00,below_lower_limit,0.447088,5.462374,True,5.462374,2025,False
1092,2025-06-22 14:56:29.897100,2025-06-22 14:56:00,03LIC_1071,SP,2.0000,2.0000,increase,57.354620,35.637906,3.946076,...,NaN,NaT,NaT,unknown,NaN,NaN,False,NaN,2025,False
1093,2025-06-22 16:04:26.492700,2025-06-22 16:04:00,03LIC_1071,SP,2.0000,2.0000,increase,58.825974,35.860939,4.586695,...,NaN,NaT,NaT,unknown,NaN,NaN,False,NaN,2025,False
1094,2025-06-22 17:03:51.553700,2025-06-22 17:03:00,03PIC_1013,OP,-2.0000,2.0000,decrease,72.000000,42.714536,3.922006,...,NaN,NaT,NaT,unknown,NaN,NaN,False,NaN,2025,False


In [24]:
# ── Visual 1: Action PV vs settled PV, with operating limits ───────────
plot_df = response_df[response_df['source_tag'] == '03LIC_1071'].dropna(subset=['pv_at_action', 'settled_pv']).copy()

fig = px.scatter(
    plot_df,
    x='pv_at_action',
    y='settled_pv',
    color='action_direction',
    size='abs_op_delta',
    hover_data=[
        'source_tag', 'event_time', 'change_type', 'op_delta',
        'response_time_mins', 'settling_time_mins', 'action_zone',
        'settled_zone', 'movement_towards_limits'
    ],
    title='03LIC_1071.PV At 03LIC_1071 Action Time Versus Settled Value',
    labels={
        'pv_at_action': '03LIC_1071.PV when action was taken',
        'settled_pv': '03LIC_1071.PV settled around',
        'abs_op_delta': '|OP change|'
    }
)

min_axis = min(plot_df['pv_at_action'].min(), plot_df['settled_pv'].min(), TARGET_LOWER_LIMIT) - 1
max_axis = max(plot_df['pv_at_action'].max(), plot_df['settled_pv'].max(), TARGET_UPPER_LIMIT) + 1

fig.add_shape(
    type='rect',
    x0=min_axis, x1=max_axis,
    y0=TARGET_LOWER_LIMIT, y1=TARGET_UPPER_LIMIT,
    fillcolor='lightgreen', opacity=0.15, line_width=0,
    layer='below'
    )
fig.add_shape(
    type='rect',
    x0=TARGET_LOWER_LIMIT, x1=TARGET_UPPER_LIMIT,
    y0=min_axis, y1=max_axis,
    fillcolor='lightgreen', opacity=0.15, line_width=0,
    layer='below'
    )
fig.add_shape(
    type='line',
    x0=min_axis, y0=min_axis, x1=max_axis, y1=max_axis,
    line=dict(color='gray', dash='dash')
    )
fig.add_hline(y=TARGET_LOWER_LIMIT, line_dash='dot', line_color='firebrick')
fig.add_hline(y=TARGET_UPPER_LIMIT, line_dash='dot', line_color='firebrick')
fig.add_hline(y=71, line_dash='dot', line_color='green')
fig.add_hline(y=28.75, line_dash='dot', line_color='red')
fig.add_vline(x=28.75, line_dash='dot', line_color='red')
fig.add_vline(x=TARGET_LOWER_LIMIT, line_dash='dot', line_color='firebrick')
fig.add_vline(x=TARGET_UPPER_LIMIT, line_dash='dot', line_color='firebrick')
fig.add_vline(x=71, line_dash='dot', line_color='green')

fig.update_layout(height=650)
fig.show()

In [23]:
# ── Visual 1: Action PV vs settled PV, with operating limits ───────────
plot_df = response_df[response_df['source_tag'] == '03LIC_1016'].dropna(subset=['pv_at_action', 'settled_pv']).copy()

fig = px.scatter(
    plot_df,
    x='pv_at_action',
    y='settled_pv',
    color='action_direction',
    size='abs_op_delta',
    hover_data=[
        'source_tag', 'event_time', 'change_type', 'op_delta',
        'response_time_mins', 'settling_time_mins', 'action_zone',
        'settled_zone', 'movement_towards_limits'
    ],
    title='03LIC_1071.PV At 03LIC_1016 Action Time Versus Settled Value',
    labels={
        'pv_at_action': '03LIC_1071.PV when action was taken',
        'settled_pv': '03LIC_1071.PV settled around',
        'abs_op_delta': '|OP change|'
    }
)

min_axis = min(plot_df['pv_at_action'].min(), plot_df['settled_pv'].min(), TARGET_LOWER_LIMIT) - 1
max_axis = max(plot_df['pv_at_action'].max(), plot_df['settled_pv'].max(), TARGET_UPPER_LIMIT) + 1

fig.add_shape(
    type='rect',
    x0=min_axis, x1=max_axis,
    y0=TARGET_LOWER_LIMIT, y1=TARGET_UPPER_LIMIT,
    fillcolor='lightgreen', opacity=0.15, line_width=0,
    layer='below'
    )
fig.add_shape(
    type='rect',
    x0=TARGET_LOWER_LIMIT, x1=TARGET_UPPER_LIMIT,
    y0=min_axis, y1=max_axis,
    fillcolor='lightgreen', opacity=0.15, line_width=0,
    layer='below'
    )
fig.add_shape(
    type='line',
    x0=min_axis, y0=min_axis, x1=max_axis, y1=max_axis,
    line=dict(color='gray', dash='dash')
    )
fig.add_hline(y=TARGET_LOWER_LIMIT, line_dash='dot', line_color='firebrick')
fig.add_hline(y=TARGET_UPPER_LIMIT, line_dash='dot', line_color='firebrick')
fig.add_hline(y=71, line_dash='dot', line_color='green')
fig.add_hline(y=28.75, line_dash='dot', line_color='red')
fig.add_vline(x=28.75, line_dash='dot', line_color='red')
fig.add_vline(x=TARGET_LOWER_LIMIT, line_dash='dot', line_color='firebrick')
fig.add_vline(x=TARGET_UPPER_LIMIT, line_dash='dot', line_color='firebrick')
fig.add_vline(x=71, line_dash='dot', line_color='green')

fig.update_layout(height=650)
fig.show()

In [22]:
# ── Visual 1: Action PV vs settled PV, with operating limits ───────────
plot_df = response_df[response_df['source_tag'] == '03PIC_1013'].dropna(subset=['pv_at_action', 'settled_pv']).copy()

fig = px.scatter(
    plot_df,
    x='pv_at_action',
    y='settled_pv',
    color='action_direction',
    size='abs_op_delta',
    hover_data=[
        'source_tag', 'event_time', 'change_type', 'op_delta',
        'response_time_mins', 'settling_time_mins', 'action_zone',
        'settled_zone', 'movement_towards_limits'
    ],
    title='03LIC_1071.PV At 03PIC_1013 Action Time Versus Settled Value',
    labels={
        'pv_at_action': '03LIC_1071.PV when action was taken',
        'settled_pv': '03LIC_1071.PV settled around',
        'abs_op_delta': '|OP change|'
    }
)

min_axis = min(plot_df['pv_at_action'].min(), plot_df['settled_pv'].min(), TARGET_LOWER_LIMIT) - 1
max_axis = max(plot_df['pv_at_action'].max(), plot_df['settled_pv'].max(), TARGET_UPPER_LIMIT) + 1

fig.add_shape(
    type='rect',
    x0=min_axis, x1=max_axis,
    y0=TARGET_LOWER_LIMIT, y1=TARGET_UPPER_LIMIT,
    fillcolor='lightgreen', opacity=0.15, line_width=0,
    layer='below'
    )
fig.add_shape(
    type='rect',
    x0=TARGET_LOWER_LIMIT, x1=TARGET_UPPER_LIMIT,
    y0=min_axis, y1=max_axis,
    fillcolor='lightgreen', opacity=0.15, line_width=0,
    layer='below'
    )
fig.add_shape(
    type='line',
    x0=min_axis, y0=min_axis, x1=max_axis, y1=max_axis,
    line=dict(color='gray', dash='dash')
    )
fig.add_hline(y=TARGET_LOWER_LIMIT, line_dash='dot', line_color='firebrick')
fig.add_hline(y=TARGET_UPPER_LIMIT, line_dash='dot', line_color='firebrick')
fig.add_hline(y=71, line_dash='dot', line_color='green')
fig.add_hline(y=28.75, line_dash='dot', line_color='red')
fig.add_vline(x=28.75, line_dash='dot', line_color='red')
fig.add_vline(x=TARGET_LOWER_LIMIT, line_dash='dot', line_color='firebrick')
fig.add_vline(x=TARGET_UPPER_LIMIT, line_dash='dot', line_color='firebrick')
fig.add_vline(x=71, line_dash='dot', line_color='green')

fig.update_layout(height=650)
fig.show()

In [8]:
# ── Visual 2: Settling time by tag and action direction ────────────────
plot_df = response_df.dropna(subset=['settling_time_mins']).copy()
tag_counts = plot_df['source_tag'].value_counts()
top_tags = tag_counts[tag_counts >= 10].index.tolist()
plot_df = plot_df[plot_df['source_tag'].isin(top_tags)]

fig = px.box(
    plot_df,
    x='source_tag',
    y='settling_time_mins',
    color='action_direction',
    points='all',
    hover_data=[
        'event_time', 'change_type', 'op_delta', 'pv_at_action',
        'settled_pv', 'action_zone', 'settled_zone', 'movement_towards_limits'
    ],
    title='Settling Time By Tag, Split By Action Direction',
    labels={
        'source_tag': 'Source tag changed',
        'settling_time_mins': 'Settling time (minutes)'
    }
)
fig.add_hrect(
    y0=0, y1=15,
    fillcolor='lightgreen', opacity=0.08, line_width=0
    )
fig.update_layout(height=550, xaxis_tickangle=45)
fig.show()